# Round 0 · Explore the Trailmates log

Trailmates is Kestrel Outfitters' hobby community: members message each other about trips, sell used
gear on a swap board and pay through the community wallet. You are its trust-and-safety team, and you
have ninety days of its records. Somewhere in them, crews are working members for money.

**What you have is metadata, not messages.** A message is a row: who, to whom, when, a length bucket,
and whether it carried a link, a payout reference or an "add me on another app" handle. There is no
message text anywhere in this lab. That is deliberate: the chapter argues that the scam operation's
costs show in the metadata, and that reading members' private messages is a decision with its own
legal and trust price.

Nothing is scored in this notebook. Look around, form hunches, and write them down.

In [ ]:
import sys
sys.path.insert(0, "../tools")
import pandas as pd
import trailmates as tm
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

F = tm.frames()
mem, dms, pays, reps, feats = F["members"], F["dms"], F["payments"], F["reports"], F["features"]
print(f"{len(mem)} members, {len(dms)} message events, {len(pays)} payments, {len(reps)} reports")

## The tables

- `mem`: one row per member: when they joined, the network (ASN) and device they signed up from,
  their photo (as a hash), declared and observed regions, verification flags.
- `dms`: one row per message event (see the columns below).
- `pays`: one row per wallet payment, with the facts the platform knew **at that moment**
  (first payment to this payee? how old is the payee? did a recent contact ask to move apps?).
- `reps`: member reports (`scam`, `spam`, `harassment`, `other`).
- `feats`: per-member features computed from all of the above by `tools/trailmates.py`.

In [ ]:
dms.head()

In [ ]:
kinds = {a["asn"]: a["kind"] for a in tm.world()["asns"]}
mem["egress"] = mem["signup_asn"].map(kinds)
mem["egress"].value_counts()

In [ ]:
# when does the community talk? (UTC hour of every message)
dms.groupby(dms["ts"].dt.hour).size().plot.bar(figsize=(10, 3), title="messages per UTC hour");

In [ ]:
feats.describe().T[["mean", "50%", "max"]]

## Questions to sit with (no answers here; the rounds come back to each)

1. Who opens the most conversations? Sort by `outbound_first` and `fanout_48h`. What else do the top accounts share?
2. Some photos are used by more than one account (`photo_count_raw`). Which photos, and by whom?
3. `hour_concentration` is the share of an account's messages inside its busiest ten UTC hours. What
   kind of member sends everything inside one ten-hour block, seven days a week?
4. What does a first payment to a brand-new contact usually look like here, and how long after they met?
5. Look at `tm.intel()`, the partner bank's list. Who mentioned those handles and domains?

In [ ]:
feats.sort_values("fanout_48h", ascending=False).head(15)[
    ["age_at_first_outbound_days", "fanout_48h", "outbound_first", "inbound_first", "photo_count_raw", "egress"]]

In [ ]:
pd.DataFrame(tm.intel())

Next: `01_profile.ipynb`, Round 1.